# Amazon Product Substitute Classification

This notebook builds a machine learning classifier to predict whether one Amazon product can serve as a substitute for another product.

## Objective
Given a pair of products, the goal is to classify whether the candidate product is a substitute for the reference product. The workflow includes exploratory data analysis, feature selection, preprocessing, neural network training, validation, threshold tuning, and prediction generation.

## Tools Used
- Python
- pandas and NumPy
- scikit-learn pipelines and transformers
- PyTorch
- Jupyter Notebook


## 1. Load the Data

Load the training and test datasets into pandas DataFrames and inspect their shapes and sample rows.


The training dataset contains labeled product pairs, while the test dataset contains product pairs for prediction.


In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

training_data = pd.read_csv('training.csv')
test_data = pd.read_csv('public_test_features.csv')

print('The shape of the training dataset is:', training_data.shape)
print('The shape of the test dataset is:', test_data.shape)

display(training_data.head())
display(test_data.head())

## 2. Exploratory Data Analysis

Inspect the structure of the training and test datasets, including data types, missing values, label distribution, and feature summaries.


In [ ]:
display(training_data.head())
print("Training shape:", training_data.shape)

print("\nDtypes (count):")
print(training_data.dtypes.value_counts())

print("\nLabel distribution:")
print(training_data['label'].value_counts(dropna=False))

print("\nTop 15 columns with missing values:")
missing_train = training_data.isnull().sum().sort_values(ascending=False)
print(missing_train.head(15))

print("\nNumeric summary (first 12 rows):")
display(training_data.describe().T.head(12))

cat_cols = [c for c in training_data.columns if training_data[c].dtype == 'object']
uniq_counts = training_data[cat_cols].nunique().sort_values(ascending=False)
print("\nTop 12 high-cardinality categorical columns:")
print(uniq_counts.head(12))

import matplotlib.pyplot as plt
training_data['label'].value_counts().plot(kind='bar')
plt.title("Target label counts")
plt.xlabel("label")
plt.ylabel("count")
plt.show()


In [ ]:
display(test_data.head())
print("Test shape:", test_data.shape)

print("\nDtypes (count):")
print(test_data.dtypes.value_counts())

print("\nTop 15 columns with missing values:")
missing_test = test_data.isnull().sum().sort_values(ascending=False)
print(missing_test.head(15))

print("\nNumeric summary (first 12 rows):")
display(test_data.describe().T.head(12))

cat_cols_test = [c for c in test_data.columns if test_data[c].dtype == 'object']
uniq_counts_test = test_data[cat_cols_test].nunique().sort_values(ascending=False)
print("\nTop 12 high-cardinality categorical columns:")
print(uniq_counts_test.head(12))

train_cols = set(training_data.columns) - {'label'}
test_cols = set(test_data.columns)
diff = sorted(list(train_cols - test_cols))
print("\nColumns in TRAIN (excluding 'label') but not in TEST:")
print(diff[:25], ("... (+ more)" if len(diff) > 25 else ""))


### Dataset Features

Review the available feature columns in the training and test datasets.


In [ ]:
print("Training data features")
print("Number of features:", len(training_data.columns))
print("Feature names (first 20):", training_data.columns.tolist()[:20], "...")

print("\nTest data features")
print("Number of features:", len(test_data.columns))
print("Feature names (first 20):", test_data.columns.tolist()[:20], "...")



## 3. Feature Selection

Select numerical, categorical, and text-based features from both the reference product and candidate product. These features are used to model product similarity and substitute relationships.


In [ ]:

numerical_features = [
    "key_item_package_quantity", "key_item_height", "key_item_width",
    "key_item_length", "key_item_weight",
    "key_pkg_height", "key_pkg_width", "key_pkg_length", "key_pkg_weight",
    "cand_item_package_quantity", "cand_item_height", "cand_item_width",
    "cand_item_length", "cand_item_weight",
    "cand_pkg_height", "cand_pkg_width", "cand_pkg_length", "cand_pkg_weight"
]

categorical_features = [
    "key_classification_code", "key_has_ean", "key_has_online_play",
    "cand_classification_code", "cand_has_ean", "cand_has_online_play"
]

text_features = ["key_item_name", "cand_item_name"]

model_features = numerical_features + categorical_features + text_features

model_target = "label"

print("Number of selected features:", len(model_features))
print("Sample features:", model_features[:10], "...")


## 4. Train/Validation Split

Split the labeled data into training and validation sets to evaluate model performance during development.


In [ ]:
from sklearn.model_selection import train_test_split

X = training_data[model_features]
y = training_data[model_target]

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  
)

print("Train set shape:", X_train.shape)
print("Validation set shape:", X_val.shape)
print("\nTarget distribution in training set:\n", y_train.value_counts(normalize=True))
print("\nTarget distribution in validation set:\n", y_val.value_counts(normalize=True))


## 5. Preprocessing Pipeline

Use scikit-learn pipelines and a ColumnTransformer to process numerical, categorical, and text features consistently across training, validation, and test data.


In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
import pandas as pd

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False))
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

def _fillna_flat_1d(X):
    if isinstance(X, pd.DataFrame):
        X = X.iloc[:, 0]
    return X.fillna("").astype(str).values

text_fill_flat = FunctionTransformer(_fillna_flat_1d, validate=False, feature_names_out="one-to-one")

text_pipes = []
for col in text_features:
    text_pipes.append(
        (f"tfidf_{col}",
         Pipeline([
             ("fill_and_flatten", text_fill_flat),
             ("tfidf", TfidfVectorizer(
                 ngram_range=(1, 2),
                 min_df=5,
                 max_features=3000
             ))
         ]),
         col)
    )

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipe, numerical_features),
        ("cat", cat_pipe, categorical_features),
        *text_pipes
    ],
    remainder="drop"
)

preprocessor.fit(X_train)

X_train_proc = preprocessor.transform(X_train)
X_val_proc   = preprocessor.transform(X_val)
X_test_proc  = preprocessor.transform(test_data[model_features])

def _shape_info(name, X):
    return f"{name} -> shape={X.shape}, nnz={X.nnz}" if sp.issparse(X) else f"{name} -> shape={X.shape}"

print(_shape_info("X_train_proc", X_train_proc))
print(_shape_info("X_val_proc",   X_val_proc))
print(_shape_info("X_test_proc",  X_test_proc))


## 6. Neural Network Training and Validation

Train a neural network classifier in PyTorch and evaluate performance using validation accuracy, F1 score, and ROC-AUC.


In [ ]:
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [ ]:

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

class CSRDataset(Dataset):
    def __init__(self, X_csr, y=None):
        self.X = X_csr
        self.y = None if y is None else np.asarray(y, dtype=np.float32)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        x = self.X[idx].toarray().ravel().astype(np.float32)
        if self.y is None:
            return torch.from_numpy(x)
        return torch.from_numpy(x), torch.tensor(self.y[idx])

input_dim = X_train_proc.shape[1]

class MLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP(input_dim).to(device)

train_loader = DataLoader(CSRDataset(X_train_proc, y_train.values),
                          batch_size=1024, shuffle=True, num_workers=0)
val_loader   = DataLoader(CSRDataset(X_val_proc,   y_val.values),
                          batch_size=2048, shuffle=False, num_workers=0)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_f1 = 0.0
best_state = None
patience = 2
epochs = 8

def sigmoid_np(a):
    return 1.0 / (1.0 + np.exp(-a))

for epoch in range(1, epochs + 1):
    model.train()
    running = 0.0
    n = 0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running += loss.item() * xb.size(0)
        n += xb.size(0)
    train_loss = running / max(n, 1)


    model.eval()
    all_logits = []
    all_y = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            logits = model(xb)
            all_logits.append(logits.cpu())
            all_y.append(yb)
    val_logits = torch.cat(all_logits).numpy()
    val_y = torch.cat(all_y).numpy()
    val_probs = sigmoid_np(val_logits)
    val_pred  = (val_probs >= 0.5).astype(int)

    acc = accuracy_score(val_y, val_pred)
    f1  = f1_score(val_y, val_pred)
    auc = roc_auc_score(val_y, val_probs)

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | "
          f"val_acc={acc:.4f} | val_f1={f1:.4f} | val_auc={auc:.4f}")

   
    if f1 > best_f1:
        best_f1 = f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_left = patience
    else:
        patience_left -= 1
        if patience_left <= 0:
            print("Early stopping.")
            break


if best_state is not None:
    model.load_state_dict(best_state)
    model.to(device)

print("Best validation F1:", best_f1)


## 7. Test Set Predictions

Use the trained model and optimized classification threshold to generate predictions for the test dataset.


In [ ]:

from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
import numpy as np
import torch

val_loader  = DataLoader(CSRDataset(X_val_proc,  y_val.values), batch_size=4096, shuffle=False, num_workers=0)
test_loader = DataLoader(CSRDataset(X_test_proc),                 batch_size=4096, shuffle=False, num_workers=0)

def _sigmoid_np(a):
    return 1.0 / (1.0 + np.exp(-a))

model.eval()
val_logits_list, val_y_list = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        logits = model(xb)
        val_logits_list.append(logits.cpu().numpy())
        val_y_list.append(yb.numpy())

val_logits = np.concatenate(val_logits_list)
val_y_true = np.concatenate(val_y_list)
val_probs  = _sigmoid_np(val_logits)

ths = np.linspace(0.2, 0.8, 25)
best_t, best_f1 = max(((t, f1_score(val_y_true, (val_probs >= t).astype(int))) for t in ths),
                      key=lambda x: x[1])
print(f"Best threshold on validation = {best_t:.3f} (F1={best_f1:.4f})")

test_logits_list = []
with torch.no_grad():
    for xb in test_loader:
        xb = xb.to(device)
        logits = model(xb)
        test_logits_list.append(logits.cpu().numpy())

test_logits = np.concatenate(test_logits_list)
test_probs  = _sigmoid_np(test_logits)
test_predictions = (test_probs >= best_t).astype(int)
print("Test predictions shape:", test_predictions.shape)


## 8. Export Predictions

Save the final predictions to a CSV file for downstream evaluation or submission.


In [ ]:

import pandas as pd

if 'test_predictions' in globals():
    preds = pd.Series(test_predictions).astype(int)
elif 'logreg' in globals():
    preds = pd.Series(logreg.predict(X_test_proc)).astype(int)
else:
    raise NameError("No test predictions found. Run Section 4 (or train a model) to create predictions.")

submission = pd.DataFrame({
    "ID": test_data["ID"].astype(int),
    "label": preds
})

assert len(submission) == len(test_data), "Prediction count does not match test rows."
assert set(submission["label"].unique()) <= {0, 1}, "Labels must be 0/1 integers."

submission.to_csv("submission.csv", index=False)
print("Saved -> submission.csv")
print("Submission shape:", submission.shape)
print("Label counts:\n", submission["label"].value_counts())

submission.head()
